# MCDO Covariance Determinants

Quick check of Monte Carlo Dropout covariance determinants for the MNIST sweep.


In [5]:
from pathlib import Path
import torch
import numpy as np

# Select which dataset of covariance matrices to inspect.
DATASET_RELATIVE = Path("runs/sim2_noise_study/mcdo/original/individual")

# Resolve the dataset location whether the kernel starts in repo root or notebooks/.
base_candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
dataset_root = None
for base in base_candidates:
    candidate = (base / DATASET_RELATIVE).resolve()
    if candidate.exists():
        dataset_root = candidate
        break

if dataset_root is None:
    raise FileNotFoundError(f"Could not locate {DATASET_RELATIVE} from the current working directory.")

# Collect determinants for each saved covariance matrix.
det_records = []
for sample_dir in sorted(dataset_root.iterdir()):
    sigma = torch.load(sample_dir / "Sigma.pt").cpu().numpy().astype(np.float64)
    det_records.append((float(np.linalg.det(sigma)), sample_dir.name))

count = len(det_records)
if not count:
    raise RuntimeError('No covariance matrices found')

records_sorted = sorted(det_records, key=lambda item: item[0])
mid = count // 2
min_det, min_idx = records_sorted[0]
median_det, median_idx = records_sorted[mid]
max_det, max_idx = records_sorted[-1]

def describe(sample_id: str, label: str) -> None:
    sigma = torch.load(dataset_root / sample_id / "Sigma.pt").cpu().numpy().astype(np.float64)
    pinv = np.linalg.pinv(sigma)
    recon = sigma @ pinv @ sigma
    residual = np.linalg.norm(recon - sigma)
    eigvals = np.linalg.eigvalsh(sigma)
    nonzero = eigvals[eigvals > 1e-9]
    pseudo_logdet = float(np.sum(np.log(nonzero))) if nonzero.size else float('-inf')
    print(f"\n{label} sample {sample_id}")
    print(f"  det: {float(np.linalg.det(sigma))}")
    print(f"  pseudo-logdet (>1e-9 eigs): {pseudo_logdet}")
    print(f"  retained eigenvalues: {nonzero.size}")
    print(f"  pinv residual ||Σ Σ⁺ Σ - Σ||_F: {residual:.3e}")
    print(f"  pinv Frobenius norm: {np.linalg.norm(pinv):.3e}")

print(f"Using data under: {dataset_root}")
print(f"Samples analysed: {count}")
print(f"Min determinant: {min_det} (sample {min_idx})")
print(f"Median determinant: {median_det} (sample {median_idx})")
print(f"Max determinant: {max_det} (sample {max_idx})")
for label, sample in (("Min", min_idx), ("Median", median_idx), ("Max", max_idx)):
    describe(sample, label)


Using data under: /home/fwromano/Documents/Code/UCLIP/runs/sim2_noise_study/mcdo/original/individual
Samples analysed: 46
Min determinant: 0.0 (sample 00000)
Median determinant: 0.0 (sample 00023)
Max determinant: 0.0 (sample 00045)

Min sample 00000
  det: 0.0
  pseudo-logdet (>1e-9 eigs): -3956.4874218096784
  retained eigenvalues: 267
  pinv residual ||Σ Σ⁺ Σ - Σ||_F: 2.487e-07
  pinv Frobenius norm: 5.747e+10

Median sample 00023
  det: 0.0
  pseudo-logdet (>1e-9 eigs): -3950.410382645895
  retained eigenvalues: 272
  pinv residual ||Σ Σ⁺ Σ - Σ||_F: 4.570e-07
  pinv Frobenius norm: 6.475e+10

Max sample 00045
  det: 0.0
  pseudo-logdet (>1e-9 eigs): -3995.502592983311
  retained eigenvalues: 274
  pinv residual ||Σ Σ⁺ Σ - Σ||_F: 8.845e-07
  pinv Frobenius norm: 4.979e+10
